# Diabetes Prediction with Logistic Regression



In [5]:
import pandas as pd

DATA_PATH = "diabetes_prediction_dataset.csv"
df = pd.read_csv(DATA_PATH)

print("First five rows:")
display(df.head())

print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())
print("Data types:")
display(df.dtypes.to_frame(name="dtype"))

print("Missing values by column:")
display(df.isna().sum().to_frame(name="missing_values"))
print("Duplicate rows:", df.duplicated().sum())

print("Samples in each diabetes class:")
display(
    df["diabetes"]
    .value_counts()
    .sort_index()
    .rename_axis("diabetes")
    .to_frame(name="sample_count")
)

First five rows:


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


Dataset shape: (100000, 9)
Column names: ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']
Data types:


,dtype
gender,object
age,float64
hypertension,int64
heart_disease,int64
smoking_history,object
bmi,float64
HbA1c_level,float64
blood_glucose_level,int64
diabetes,int64


Missing values by column:


,missing_values
gender,0
age,0
hypertension,0
heart_disease,0
smoking_history,0
bmi,0
HbA1c_level,0
blood_glucose_level,0
diabetes,0


Duplicate rows: 3854
Samples in each diabetes class:


,sample_count
diabetes,
0,91500
1,8500


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

X = df.drop(columns="diabetes")
y = df["diabetes"]

categorical_features = ["gender", "smoking_history"]
numerical_features = [
    column for column in X.columns if column not in categorical_features
]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", StandardScaler(), numerical_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Numerical features:", numerical_features)
print("Categorical features (one-hot encoded):", categorical_features)

Training rows: 80000
Testing rows: 20000
Numerical features: ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level']
Categorical features (one-hot encoded): ['gender', 'smoking_history']


The categorical columns are one-hot encoded and numerical columns are standardized inside a `ColumnTransformer`. Keeping the transformer in the model pipeline means `fit` learns the scaling statistics and category levels from `X_train` only; `X_test` is transformed using those learned values. Stratification keeps the class proportions similar in both splits.

In [7]:
from sklearn.linear_model import LogisticRegression

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Predictions (first 10):", y_pred[:10])

Predictions (first 10): [0 0 0 0 0 0 0 1 0 1]


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print(f"Accuracy: {accuracy:.4f} ({accuracy:.2%})")
print("Confusion matrix (rows = actual [0, 1], columns = predicted [0, 1]):")
print(cm)
print(f"True Negative (TN): {tn}")
print(f"False Positive (FP): {fp}")
print(f"False Negative (FN): {fn}")
print(f"True Positive (TP): {tp}")
print(f"Diabetic patients correctly classified: {tp}")
print(f"Diabetic patients incorrectly classified as non-diabetic: {fn}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["No diabetes (0)", "Diabetes (1)"],
        zero_division=0,
    )
)

Accuracy: 0.9605 (96.05%)
Confusion matrix (rows = actual [0, 1], columns = predicted [0, 1]):
[[18121   179]
 [  612  1088]]
True Negative (TN): 18121
False Positive (FP): 179
False Negative (FN): 612
True Positive (TP): 1088
Diabetic patients correctly classified: 1088
Diabetic patients incorrectly classified as non-diabetic: 612

Classification report:
                 precision    recall  f1-score   support

No diabetes (0)       0.97      0.99      0.98     18300
   Diabetes (1)       0.86      0.64      0.73      1700

       accuracy                           0.96     20000
      macro avg       0.91      0.82      0.86     20000
   weighted avg       0.96      0.96      0.96     20000



: 

## Results and interpretation

With the specified 80/20 split (`random_state=42`) and stratification, the observed accuracy is **96.045%**. The confusion matrix is `[[18121, 179], [612, 1088]]` (rows are actual labels 0/1; columns are predicted labels 0/1): TN = **18,121**, FP = **179**, FN = **612**, TP = **1,088**. Thus, **1,088** patients with diabetes were correctly classified, while **612** were incorrectly classified as non-diabetic.

Accuracy alone is not sufficient here. The dataset is imbalanced: 91,500 samples (91.5%) have label 0 and 8,500 (8.5%) have label 1. A model that always predicts 0 would already achieve 91.5% accuracy while detecting no diabetic patients. The model's diabetes-class recall is about 64% (1,088 of 1,700 diabetic test cases), so consider the confusion matrix and class-specific precision/recall as well as accuracy. The duplicate rows are reported above but retained for this requested baseline.